In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client_travel = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools_travel = await client_travel.get_tools()

In [ ]:
client_venue = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/Wedding_venue_server.py"],
            }
    }
)
# get tools
tools_venue = await client_venue.get_tools()


# get prompts
prompt_venue = await client_venue.get_prompt("local_server", "prompt")
prompt_venue = prompt_venue[0].content

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain.tools import tool

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [ ]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    genre: str
    origin: str
    destination: str
    capacity: str

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

travel_agent = create_agent(
    "gpt-5-nano",
    tools=tools_travel,
    checkpointer=InMemorySaver(),
    system_prompt="""You are a expert travel agent . You will not ask follow up questions. You wll search for flight to the desired destination from your origin destination.
    Here are your following criterias for finding the best flight: 
    price(lowest) 
    duration (shortest) 
    date (best time of the year to do the wedding)
    You will only look for one way ticket, you may also need to make multiple searches to iteratively find the best options.
    Also, you will only have origin and destination, no extra information, so think very carrefully.
    """
)

In [ ]:
venue_agent = create_agent(
    "gpt-5-nano",
    tools=tools_venue,
    checkpointer=InMemorySaver(),
    system_prompt=prompt_venue
)


In [ ]:
dj_agent = create_agent(
    "gpt-5-nano",
    tools=[query_playlist_db],
    checkpointer=InMemorySaver(),
    system_prompt="""You are a playlist expert. 
    Query the SQL Database and curate the perfect playlist based on the given genre. Once the playlist is available, give the cost and the duration of that playlist.
    Each song has a price. If you run into error when querying the database, try changing the query parameters.
    Make sure to have a list of song.
    It's an SQLite database, so before writting queries, make sure to discover the schema."""
)


In [ ]:
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent search for flights to your destination from your origin position"""
    try:
        origin = runtime.state["origin"] 
        destination = runtime.state["destination"]
    except KeyError:
        return "No favourite origin and destination found in state"
        
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"find flights from {origin} to {destination}")]},{"configurable": {"thread_id": "1"}})
    return response["messages"][-1].content

@tool
async def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent search for a wedding venue with the specified destination and capacity"""
    try:
        capacity = runtime.state["capacity"] 
        destination = runtime.state["destination"]
    except KeyError:
        return "No favourite capacity and destination found in state"
        
    response = await venue_agent.ainvoke({"messages": [HumanMessage(content=f"Find the best wedding venue at {destination} with a capacity of {capacity}")]},{"configurable": {"thread_id": "2"}})
    return response["messages"][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """DJ agent search for a playlist matching the right genre"""
    try:
        genre = runtime.state["genre"] 
    except KeyError:
        return "No favourite genre found in state"
    response = dj_agent.invoke({"messages": [HumanMessage(content=f"Find {genre} tracks for wedding playlist")]},{"configurable": {"thread_id": "3"}})
    return response['messages'][-1].content


@tool
def update_state(origin: str, destination: str, capacity: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, capacity, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "capacity": capacity, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )

In [ ]:
## Creating the main agent

coordinator_agent = create_agent(
    model='gpt-5-nano',
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""You are a wedding Coordinator. Your methodology is the following:
    -> First, find all the information you need to update the state. Then, update the state.
    -> Once that is done and returned. You will delegate tasks to your specialists for flights, venues and playlists.
    -> Finally, when you get their answers, coordinate the perfect wedding for me"""
)

In [ ]:
#Test

question = "I'm from Paris and I'd like a wedding in Gennevilliers for a capacity of 100 guests, Hispanic-genre music"

response = await coordinator_agent.ainvoke({"messages": [HumanMessage(content=question)]})

In [ ]:
from pprint import pprint

pprint(response)